# Weather EDA — SFO NOAA Hourly Data

This notebook explores the cleaned hourly SFO NOAA weather data and uses the following workflow:

1. validate and order the time-series data
2. create chronological train, validate, and test splits before EDA
3. perform EDA on the training split only so validation and test observations remain unseen.

In [1]:
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import TimeSeriesSplit

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.3f}".format)
sns.set_theme(style="whitegrid", context="notebook")

RANDOM_STATE = 207 # DATASCI
SPLIT_COLORS = {
    "train": "#4c78a8",
    "validation": "#f2a541",
    "test": "#e45756",
}

### Load the cleaned hourly data

In [2]:
REPO_ROOT = Path.cwd().parents[1]
CLEANED_PARQUET_PATH = REPO_ROOT / "Data_Preprocessing" / "santi" / "data" / "cleaned" / "sfo_noaa_cleaned.parquet"

required_columns = {
    "DATE",
    "Year",
    "Month",
    "Day",
    "Hour",
    "Minute",
    "LATITUDE",
    "LONGITUDE",
    "ELEVATION",
    "temperature",
    "dew_point_temperature",
    "relative_humidity",
    "visibility",
    "wind_speed",
    "wind_direction",
    "sea_level_pressure",
    "station_level_pressure",
    "wet_bulb_temperature",
    "ceiling_height",
    "altimeter",
    "sky_condition_baseht",
    "sky_cover_summation_baseht_1",
    "precipitation",
}

weather_df = pd.read_parquet(CLEANED_PARQUET_PATH)
missing_required_columns = sorted(required_columns.difference(weather_df.columns))

weather_df["DATE"] = pd.to_datetime(weather_df["DATE"], errors="raise")
weather_df = weather_df.sort_values("DATE").reset_index(drop=True)

print(f"Loaded: {CLEANED_PARQUET_PATH}")
print(f"Shape: {weather_df.shape}")
print(f"Date range: {weather_df['DATE'].min()} to {weather_df['DATE'].max()}")
display(weather_df.head())

Loaded: c:\Users\smeri\OneDrive\Desktop\MIDS Coursework\207-Summer26-FinalProject-MLModel\Data_Preprocessing\santi\data\cleaned\sfo_noaa_cleaned.parquet
Shape: (38732, 23)
Date range: 2022-01-01 00:56:00 to 2026-06-09 16:56:00


,DATE,Year,Month,Day,Hour,Minute,LATITUDE,LONGITUDE,ELEVATION,temperature,dew_point_temperature,relative_humidity,visibility,wind_speed,wind_direction,sea_level_pressure,station_level_pressure,wet_bulb_temperature,ceiling_height,altimeter,sky_condition_baseht,sky_cover_summation_baseht_1,precipitation
0,2022-01-01 00:56:00,2022,1,1,0,56,37.620,-122.366,3.000,10.600,5.000,69.000,16.093,7.200,300.000,1012.800,1012.300,7.900,22000.000,1012.900,610.000,610.000,0.000
1,2022-01-01 01:56:00,2022,1,1,1,56,37.620,-122.366,3.000,10.000,4.400,68.000,16.093,5.100,300.000,1013.300,1012.600,7.400,22000.000,1013.200,579.000,579.000,0.000
2,2022-01-01 02:56:00,2022,1,1,2,56,37.620,-122.366,3.000,9.400,4.400,71.000,16.093,5.100,290.000,1014.000,1013.600,7.100,22000.000,1014.200,NaN,NaN,0.000
3,2022-01-01 03:56:00,2022,1,1,3,56,37.620,-122.366,3.000,9.400,5.000,74.000,16.093,5.100,310.000,1014.600,1014.000,7.300,22000.000,1014.600,914.000,914.000,0.000
4,2022-01-01 04:56:00,2022,1,1,4,56,37.620,-122.366,3.000,8.900,5.000,77.000,16.093,2.600,330.000,1015.500,1015.000,7.100,22000.000,1015.600,NaN,NaN,0.000


### Investigating Time Gaps

In [3]:
timestamp_deltas = weather_df["DATE"].diff().dropna()
gaps_longer_than_hour = timestamp_deltas[timestamp_deltas > pd.Timedelta(hours=1)]
hourly_gap_count = len(gaps_longer_than_hour)

gap_distribution = (
    gaps_longer_than_hour.value_counts()
    .sort_index()
    .rename_axis("gap_length")
    .reset_index(name="gap_count")
)
gap_distribution["gap_hours"] = (
    gap_distribution["gap_length"] / pd.Timedelta(hours=1)
)

print(f"Gaps longer than one hour: {hourly_gap_count}")
display(gap_distribution)

Gaps longer than one hour: 53


,gap_length,gap_count,gap_hours
0,0 days 02:00:00,39,2.000
1,0 days 03:00:00,8,3.000
2,0 days 04:00:00,3,4.000
3,0 days 05:00:00,1,5.000
4,0 days 08:00:00,1,8.000
5,3 days 19:00:00,1,91.000


## 3. Chronological train, validation, and test design

The final test period begins on January 1, 2025 and remains outside cross-validation. The train period contains 2022–2024. Within it, `TimeSeriesSplit` creates two windows splits with a holdout size equal to the number of observations in 2024. This makes the final split align with the intended calendar design:

- **training:** 2022–2023
- **validation:** 2024
- **final test:** 2025 through July 2026.

In [4]:
VALIDATION_START = pd.Timestamp("2024-01-01")
TEST_START = pd.Timestamp("2025-01-01")

train_validate_df = weather_df.loc[weather_df["DATE"] < TEST_START].copy().reset_index(drop=True)
weather_test_df = weather_df.loc[weather_df["DATE"] >= TEST_START].copy().reset_index(drop=True)

validation_size = int(train_validate_df["DATE"].ge(VALIDATION_START).sum())
time_series_cv = TimeSeriesSplit(
    n_splits=2,
    test_size=validation_size,
    gap=0,
)
cv_splits = list(time_series_cv.split(train_validate_df))